# A5：Merge 匝道汇入 —— 横纵向协同控制（运动学演示）
# A5: Merge — Lateral & Longitudinal Control (Kinematic Demo)

> ⚠️ 本 notebook 为**纯 Python 运动学演示**：自车与主路车辆均为运动学模型，
> 未调用 esmini 引擎。用于理解汇入控制逻辑，真 esmini 场景见挑战任务。

## 一、实验原理 | Principle

### 1.1 什么是匝道汇入？
车辆从匝道（加速车道）汇入主路，是自动驾驶的难点之一。需要同时解决：
- **纵向**：加速到与主路车流相当的速度，找到合适的间隙
- **横向**：平滑地从匝道变道到主路，不能急打方向

### 1.2 间隙选择（Gap Selection）
汇入前必须找到前后车都安全的"窗口"：
- 与前车距离 > 安全距离（跟驰）
- 与后车距离 > 安全距离（不被追尾）
- 实际系统中还会预测后车是否会让行

### 1.3 横纵向解耦控制
本实验用简单的解耦策略：
- 纵向：PD 跟驰（与 A2 类似）
- 横向：比例控制 `steer = k * y_err`，向目标车道靠拢

## 二、实验目的 | Objectives
1. 理解汇入场景的横纵向协同需求
2. 实现简单的变道横向控制器
3. 观察汇入轨迹的平滑性

## 三、学生任务 | Your Tasks
1. **运行**仿真，观察汇入轨迹（x-y 平面图）
2. **调参**：横向增益从 0.05 改为 0.1，观察变道是否更果断（但会不会太猛？）
3. **思考题**：如果主路车流很密（前后间距 < 10m），还应该强行汇入吗？真实系统会怎么做？（提示：等待、减速、请求让行）

## 四、验证内容 | Verification
- [ ] 20 秒内无碰撞
- [ ] 成功到达目标车道（|y| < 0.3m）
- [ ] 轨迹平滑，无剧烈转向

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `未成功汇入` | 20秒内没到达目标车道 | 增大横向增益，或检查初始位置 |
| 轨迹振荡 | 横向增益太大 | 减小转向比例系数 |


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import KinematicBicycleModel
print('✓ 导入成功（纯运动学模型，未使用 esmini 引擎）')


## 1. 汇入控制器（横纵向）

In [ ]:
class MergeController(BaseController):
    """匝道汇入：纵向跟驰 + 横向变道"""
    def __init__(self, config=None):
        super().__init__(config)
        self.target_y = 0.0  # 目标车道
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        # 纵向：跟随前车
        if targets:
            lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
            gap_err = (abs(lead['x'] - ego['x']) - 4.5) - ego['v'] * 1.0
            a = max(-3.0, min(2.0, 0.8 * gap_err + 0.5 * (lead['v'] - ego['v'])))
        else:
            a = 0.5
        # 横向：向目标车道靠拢（简化：固定转向角）
        y_err = self.target_y - ego['y']
        steer = max(-0.1, min(0.1, 0.05 * y_err))
        return {'target_a': a, 'steer_angle': steer}
print('✓ Merge 控制器就绪')


## 2. 运动学仿真（纯 Python，未使用 esmini）

In [ ]:
# 汇入场景：自车从 y=-3 汇入主路 y=0
controller = MergeController()
ego = KinematicBicycleModel()
ego.reset(x=10.0, y=-3.0, yaw=0.0, v=12.0)
# 主路车辆：硬编码运动学函数（演示用，非 esmini 场景）
import math
def main_road_vehicle(t, x0=40, v=14.0):
    return {'x': x0 + v*t, 'y': 0.0, 'v': v}

is_collision, merge_success = False, False
history = []
for step in range(400):
    t = step * 0.05
    lead = main_road_vehicle(t)
    obs = {'ego': {'x': ego.x, 'y': ego.y, 'yaw': ego.yaw, 'v': ego.v},
           'targets': [lead], 'sim_time': t, 'dt': 0.05}
    act = controller.step(obs)
    ego.step(act['target_a'], act['steer_angle'], 0.05)
    history.append((t, ego.x, ego.y, ego.v))
    # 碰撞检测
    if abs(lead['x'] - ego.x) < 4.5 and abs(lead['y'] - ego.y) < 1.5:
        is_collision = True
        break
    # 汇入成功：到达目标车道
    if abs(ego.y) < 0.3 and t > 5:
        merge_success = True

print(f'✓ 完成：{len(history)} 步，碰撞={is_collision}，汇入成功={merge_success}')


## 3. 评测断言

In [ ]:
assert not is_collision, '发生碰撞!'
assert merge_success, '未成功汇入!'
print('✅ 评测通过！')


## 4. 可视化

In [ ]:
import matplotlib.pyplot as plt
times = [t for t, x, y, v in history]
xs = [x for t, x, y, v in history]
ys = [y for t, x, y, v in history]
plt.figure(figsize=(12, 4))
plt.plot(xs, ys, label='Ego trajectory')
plt.axhline(y=0, color='gray', linestyle='--', label='Main road')
plt.axhline(y=-3, color='gray', linestyle=':', label='Ramp')
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.title('Merge Trajectory')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()


## 🏆 挑战任务
1. 将本演示迁移到 esmini 真闭环（需 xosc 汇入场景）。
2. 加入间隙选择逻辑：选择前后车都安全的时机汇入。
